In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

# -------------------------------------------------
# Relative Attention
# -------------------------------------------------

class RelativeAttention(nn.Module):
    def __init__(self, d_model, num_heads, max_len=32):
        super().__init__()

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)

        self.relative_bias = nn.Parameter(
            torch.randn(num_heads, max_len, max_len)
        )

    def forward(self, x):
        B, T, C = x.shape

        Q = self.q(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / (self.head_dim ** 0.5)

        bias = self.relative_bias[:, :T, :T]
        scores = scores + bias.unsqueeze(0)

        attn = F.softmax(scores, dim=-1)

        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.out(out)


# -------------------------------------------------
# Tiny T5 Block
# -------------------------------------------------

class TinyT5Block(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.attn = RelativeAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, d_model)
        )

        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x


# -------------------------------------------------
# Topic-Aware Tiny T5
# -------------------------------------------------

class TopicAwareTinyT5(nn.Module):
    def __init__(self, vocab_size, d_model=32, num_heads=4):
        super().__init__()

        self.shared_embedding = nn.Embedding(vocab_size, d_model)

        self.encoder = TinyT5Block(d_model, num_heads)
        self.decoder = TinyT5Block(d_model, num_heads)

        self.lm_head = nn.Linear(d_model, vocab_size)

    def make_topic_vector(self, topic_ids):
        topic_vec = self.shared_embedding(topic_ids)

        # average topic word embeddings
        topic_vec = topic_vec.mean(dim=1)

        return topic_vec

    def forward(self, encoder_input_ids, decoder_input_ids, topic_ids):
        token_vec = self.shared_embedding(encoder_input_ids)

        topic_vec = self.make_topic_vector(topic_ids)

        topic_vec = topic_vec.unsqueeze(1).expand_as(token_vec)

        # topic-aware encoder input
        encoder_x = token_vec + topic_vec

        memory = self.encoder(encoder_x)

        decoder_x = self.shared_embedding(decoder_input_ids)

        out = self.decoder(decoder_x)

        logits = self.lm_head(out)

        return logits


# -------------------------------------------------
# Vocabulary
# -------------------------------------------------

vocab = {
    "<pad>": 0,
    "<bos>": 1,
    "patient": 2,
    "received": 3,
    "treatment": 4,
    "diabetes": 5,
    "medicine": 6,
    "doctor": 7,
    "hospital": 8,
}

id_to_word = {v: k for k, v in vocab.items()}


# -------------------------------------------------
# Example Data
# -------------------------------------------------

# input text: patient diabetes medicine hospital
encoder_input_ids = torch.tensor([
    [2, 5, 6, 8]
])

# topic words: doctor medicine hospital diabetes
topic_ids = torch.tensor([
    [7, 6, 8, 5]
])

# decoder input: <bos> patient received
decoder_input_ids = torch.tensor([
    [1, 2, 3]
])

# target summary: patient received treatment
target_ids = torch.tensor([
    [2, 3, 4]
])


# -------------------------------------------------
# Train
# -------------------------------------------------

model = TopicAwareTinyT5(vocab_size=len(vocab))

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(200):
    logits = model(
        encoder_input_ids,
        decoder_input_ids,
        topic_ids
    )

    loss = loss_fn(
        logits.view(-1, len(vocab)),
        target_ids.view(-1)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}, Loss={loss.item():.4f}")


# -------------------------------------------------
# Predict
# -------------------------------------------------

model.eval()

with torch.no_grad():
    logits = model(
        encoder_input_ids,
        decoder_input_ids,
        topic_ids
    )

prediction = logits.argmax(dim=-1)

print("\nPredicted token IDs:")
print(prediction)

print("\nPredicted Summary:")
for token_id in prediction[0]:
    print(id_to_word[token_id.item()], end=" ")

Epoch 50, Loss=0.0000
Epoch 100, Loss=0.0000
Epoch 150, Loss=0.0000
Epoch 200, Loss=0.0000

Predicted token IDs:
tensor([[2, 3, 4]])

Predicted Summary:
patient received treatment 